In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from implementations import *

In [30]:
data=pd.read_csv("dataset/x_train.csv")
y=pd.read_csv("dataset/y_train.csv")
x_test=pd.read_csv("dataset/x_test.csv")

In [31]:
data = data.set_index('Id')
x_test = x_test.set_index('Id')
print(data.shape)
data.head()

(328135, 321)


,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENUM,...,_PAREC1,_PASTAE1,_LMTACT1,_LMTWRK1,_LMTSCL1,_RFSEAT2,_RFSEAT3,_FLSHOT6,_PNEUMO2,_AIDTST3
Id,,,,,,,,,,,,,,,,,,,,,
0,53,11,11162015,11,16,2015,1100,2015015629,2015015629,NaN,...,1,1,3.0,3.0,4.0,1,1,NaN,NaN,2.0
1,33,12,12152015,12,15,2015,1200,2015004387,2015004387,1.0,...,9,9,3.0,3.0,4.0,9,9,NaN,NaN,NaN
2,20,10,10202015,10,20,2015,1100,2015005638,2015005638,1.0,...,4,2,2.0,2.0,3.0,1,1,1.0,2.0,2.0
3,42,6,6182015,6,18,2015,1100,2015004694,2015004694,NaN,...,2,2,2.0,2.0,3.0,1,1,2.0,2.0,2.0
4,24,11,11062015,11,6,2015,1100,2015004673,2015004673,1.0,...,9,9,3.0,3.0,4.0,1,1,9.0,9.0,2.0


In [32]:
y = y.set_index('Id')
print(y.shape)
y.head()

(328135, 1)


,_MICHD
Id,
0,-1
1,-1
2,-1
3,-1
4,-1


In [33]:
#Checking for missing values
nan = pd.isna(data)
nan = nan.sum(axis=0) / data.shape[0] * 100
nan.index = data.columns
nan = nan.to_frame(name="missing_perc")
nan = nan.sort_values("missing_perc", ascending=False)
nan

,missing_perc
LADULT,99.990248
COLGHOUS,99.990248
PCDMDECN,99.945449
ASERVIST,99.930516
ASDRVIST,99.930516
...,...
_FRT16,0.000000
_FRUITEX,0.000000
PERSDOC2,0.000000
FC60_,0.000000


In [34]:
#Keep only non sparse features (less than 90% missing values) and some sparse features that are important for the model

keep_sparse = [
    "INSULIN",
    "DIABEYE",
    "CVDASPRN",
    "WTCHSALT",
    "DRADVISE",
]

drop_sparse = nan.index[nan["missing_perc"] > 90 & ~nan.index.isin(keep_sparse)]

data_reduced = data.drop(columns=drop_sparse)
test_reduced = x_test.drop(columns=drop_sparse)

#Drop columns that are not useful for the model
drop_cols = [
    "SEQNO",
    "_PSU",

    # interview date / survey administration
    "IDATE",
    "FMONTH",
    "IMONTH",
    "IDAY",
    "IYEAR",
    "DISPCODE",
    "QSTVER",

    # BRFSS sampling / weighting machinery
    "_STSTR",
    "_RAWRAKE",
    "_WT2RAKE",
    "_DUALUSE",
    "_LLCPWT",
]

data_reduced = data_reduced.drop(columns=drop_cols)
test_reduced = test_reduced.drop(columns=drop_cols)

print(data_reduced.shape)
data_reduced.head()

(328135, 68)


,_STATE,MENTHLTH,HLTHPLN1,PERSDOC2,BLOODCHO,CVDSTRK3,ASTHMA3,CHCOCNCR,CHCCOPD1,ADDEPEV2,...,_PACAT1,_PAINDX1,_PA150R2,_PA300R2,_PA30021,_PASTRNG,_PAREC1,_PASTAE1,_RFSEAT2,_RFSEAT3
Id,,,,,,,,,,,,,,,,,,,,,
0,53,5,1,1,1,2,2,2,2,1,...,2,1,1,2,2,1,1,1,1,1
1,33,88,1,1,1,2,2,2,2,2,...,9,9,9,9,9,9,9,9,9,9
2,20,77,1,1,1,2,2,2,2,2,...,3,2,2,2,2,2,4,2,1,1
3,42,88,1,1,1,2,2,2,2,2,...,1,1,1,1,1,2,2,2,1,1
4,24,88,1,9,1,2,2,2,2,2,...,9,9,9,9,9,2,9,9,1,1


In [35]:
constant_cols = [col for col in data_reduced.columns if data_reduced[col].nunique() <= 1]

print("Constant columns found:", constant_cols)

# Drop constant columns
data_reduced = data_reduced.drop(columns=constant_cols)
test_reduced = test_reduced.drop(columns=constant_cols)
data_reduced.head()

Constant columns found: []


,_STATE,MENTHLTH,HLTHPLN1,PERSDOC2,BLOODCHO,CVDSTRK3,ASTHMA3,CHCOCNCR,CHCCOPD1,ADDEPEV2,...,_PACAT1,_PAINDX1,_PA150R2,_PA300R2,_PA30021,_PASTRNG,_PAREC1,_PASTAE1,_RFSEAT2,_RFSEAT3
Id,,,,,,,,,,,,,,,,,,,,,
0,53,5,1,1,1,2,2,2,2,1,...,2,1,1,2,2,1,1,1,1,1
1,33,88,1,1,1,2,2,2,2,2,...,9,9,9,9,9,9,9,9,9,9
2,20,77,1,1,1,2,2,2,2,2,...,3,2,2,2,2,2,4,2,1,1
3,42,88,1,1,1,2,2,2,2,2,...,1,1,1,1,1,2,2,2,1,1
4,24,88,1,9,1,2,2,2,2,2,...,9,9,9,9,9,2,9,9,1,1


In [36]:
data_reduced=data_reduced-data_reduced.mean()
test_reduced=test_reduced-test_reduced.mean()
data_reduced=data_reduced/data_reduced.std()
test_reduced=test_reduced/test_reduced.std()
data_reduced["intercept"]=np.ones(len(data_reduced))
test_reduced["intercept"]=np.ones(len(test_reduced))
data_reduced.head()

,_STATE,MENTHLTH,HLTHPLN1,PERSDOC2,BLOODCHO,CVDSTRK3,ASTHMA3,CHCOCNCR,CHCCOPD1,ADDEPEV2,...,_PAINDX1,_PA150R2,_PA300R2,_PA30021,_PASTRNG,_PAREC1,_PASTAE1,_RFSEAT2,_RFSEAT3,intercept
Id,,,,,,,,,,,,,,,,,,,,,
0,1.436294,-1.666924,-0.198136,-0.474657,-0.264029,0.094103,0.267552,0.231349,0.125806,-1.534356,...,-0.551977,-0.653493,-0.341088,-0.235988,-0.652207,-1.038910,-0.710654,-0.348325,-0.376114,1.0
1,0.188772,0.649439,-0.198136,-0.474657,-0.264029,0.094103,0.267552,0.231349,0.125806,0.303167,...,2.658326,2.557064,2.511043,2.591247,2.960320,2.300876,2.561214,3.048224,3.033846,1.0
2,-0.622118,0.342451,-0.198136,-0.474657,-0.264029,0.094103,0.267552,0.231349,0.125806,0.303167,...,-0.150689,-0.252173,-0.341088,-0.235988,-0.200641,0.213510,-0.301671,-0.348325,-0.376114,1.0
3,0.750157,0.649439,-0.198136,-0.474657,-0.264029,0.094103,0.267552,0.231349,0.125806,0.303167,...,-0.551977,-0.653493,-0.748535,-0.639879,-0.200641,-0.621437,-0.301671,-0.348325,-0.376114,1.0
4,-0.372613,0.649439,-0.198136,9.162834,-0.264029,0.094103,0.267552,0.231349,0.125806,0.303167,...,2.658326,2.557064,2.511043,2.591247,-0.200641,2.300876,2.561214,-0.348325,-0.376114,1.0


In [37]:
#Split the dataset into training and validation sets
x_train = data_reduced.sample(frac=0.8, random_state=21)
x_validation = data_reduced.drop(index=x_train.index)

y_train = y.loc[x_train.index]
y_validation = y.loc[x_validation.index]

In [38]:
y_train_bin = (y_train.values.ravel() == 1).astype(int)
y_validation_bin = (y_validation.values.ravel() == 1).astype(int)

# Test weighted logistic regression

In [39]:
pos_weight = len(y_train_bin) / np.sum(y_train_bin)
neg_weight = len(y_train_bin) / (len(y_train_bin) - np.sum(y_train_bin))

weights = np.where(y_train_bin == 1, pos_weight, neg_weight)

w = np.zeros(x_train.shape[1])
sigma = sigmoid(x_train @ w)
gradient = x_train.T @ (weights * (sigma - y_train_bin)) / len(y_train_bin)

In [40]:
# Parametri
max_iters = 2000
gamma = 0.01
lambda_ = 0.01
prev_loss = np.inf

# Training loop con class weighting + L2 + early stopping
for i in range(max_iters):
    # forward
    z = x_train @ w
    sigma = sigmoid(z)

    # loss logistica + L2
    loss = np.sum(np.logaddexp(0, z) - y_train_bin * z) / len(y_train_bin)
    loss += lambda_ * np.sum(w**2)

    # gradiente con class weighting + L2
    gradient = x_train.T @ (weights * (sigma - y_train_bin)) / len(y_train_bin)
    gradient += 2 * lambda_ * w

    # update
    w -= gamma * gradient

    # early stopping
    if abs(prev_loss - loss) < 1e-7:
        break

    prev_loss = loss

# --- VALIDATION ---
val_probs = sigmoid(x_validation @ w)

best_t = 0
best_f1 = 0

for t in np.linspace(0.05, 0.95, 200):
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(y_validation_bin, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Soglia ottimale:", best_t)
print("F1-score ottimale:", best_f1)

# Confusion matrix manuale
preds_val = (val_probs >= best_t).astype(int)

tp = np.sum((y_validation_bin == 1) & (preds_val == 1))
fp = np.sum((y_validation_bin == 0) & (preds_val == 1))
tn = np.sum((y_validation_bin == 0) & (preds_val == 0))
fn = np.sum((y_validation_bin == 1) & (preds_val == 0))

print("TP:", tp, "FP:", fp, "TN:", tn, "FN:", fn)

# --- TEST PREDICTION ---
test_probs = sigmoid(test_reduced @ w)
final = np.where(test_probs >= best_t, 1, -1)


Soglia ottimale: 0.6876884422110553
F1-score ottimale: 0.37868731563421826
TP: 3081 FP: 7352 TN: 52436 FN: 2758


In [41]:
final

array([-1, -1, -1, ..., -1,  1, -1], shape=(109379,))

In [42]:
print("Final predictions shape:", final.shape)

Final predictions shape: (109379,)


In [43]:
# Crea un DataFrame con Id e Prediction
submission = pd.DataFrame({
    "Id": x_test.index,
    "Prediction": final
})

# Salva il CSV nel formato richiesto
submission.to_csv("submission.csv", index=False)